## Data Preprocessing
This file contains pipelines to preprocess data for various ML methods:
- LLM/BERT
- Aspect-Based Sentiment Analysis (ABSA) with spaCy
- (Potentially) Time series

### 1. Global setup and environment

In [47]:
import pandas as pd
import spacy
import re
from bs4 import BeautifulSoup
import contractions
from tqdm.auto import tqdm # Progress bars are essential for large NLP tasks

nlp = spacy.load("en_core_web_sm")

RAW_DATA_PATH = "../data/raw/synthetic_social_media_data.csv"
PROCESSED_DIR = "../data/processed/"

### 2. Basic Cleaning for BERT

In [48]:
df = pd.read_csv(RAW_DATA_PATH)

def universal_clean(text):
    """Removes HTML tags and expands contractions (e.g., don't -> do not)."""
    if not isinstance(text, str):
        return ""
    # Strip HTML
    text = BeautifulSoup(text, "html.parser").get_text()
    # Expand contractions (crucial for accurate sentiment/negation later)
    text = contractions.fix(text)
    # Remove excessive whitespaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print("Applying universal cleaning...")
df['Clean_text'] = df['Post Content'].apply(universal_clean)


print(f"Base data loaded and cleaned: {df.shape[0]} rows.")

Applying universal cleaning...
Base data loaded and cleaned: 2000 rows.


##### Saving to file

In [49]:
bert_df = df[['Post ID', 'Clean_text', 'Sentiment Label']].copy()

bert_df = bert_df[bert_df['Clean_text'].str.len() > 0]

bert_out_path = f"{PROCESSED_DIR}bert_training_data_social_media.csv"
bert_df.to_csv(bert_out_path, index=False)

print(f"BERT data saved to {bert_out_path}")

BERT data saved to ../data/processed/bert_training_data_social_media.csv


### 3. Extract aspects for ABSA

In [50]:
def extract_aspects_with_negation(text):
    """Extracts Noun-Adjective pairs, handling basic negations."""
    doc = nlp(text)
    aspects = []
    
    for token in doc:
        if token.pos_ in ["NOUN", "PROPN"]:
            for child in token.children:
                if child.pos_ == "ADJ":
                    # Check if the adjective is negated (e.g., "not good")
                    is_negated = any(w.dep_ == "neg" for w in child.children)
                    prefix = "not " if is_negated else ""
                    
                    aspects.append({
                        "aspect": token.lemma_.lower(),
                        "descriptor": f"{prefix}{child.lemma_.lower()}"
                    })
    return aspects

print("Extracting aspects via dependency parsing (this may take a moment)...")
df['Aspects'] = df['Clean_text'].apply(extract_aspects_with_negation)

Extracting aspects via dependency parsing (this may take a moment)...


##### Saving to file

In [51]:
absa_out_path = f"{PROCESSED_DIR}absa_data_social_media.json"
df[['Post ID', 'Aspects']].to_json(absa_out_path, orient="records")

print(f"ABSA data saved to {absa_out_path}")

ABSA data saved to ../data/processed/absa_data_social_media.json


### 4. Time series analysis preparation

In [52]:
sentiment_mapping = {
    "positive": 1,
    "neutral": 0,
    "negative": -1
}

df['Numeric_sentiment'] = df['Sentiment Label'].astype(str).str.lower().map(sentiment_mapping)

df['Date'] = pd.to_datetime(df['Post Date and Time'].str[:10], errors='coerce')
ts_df = df.dropna(subset=['Date']).copy()
ts_df.set_index('Date', inplace=True)

# Resample to Daily ('D') frequency
daily_sentiment = ts_df.resample('D').agg({
    'Numeric_sentiment': 'mean',  # Calculates the average sentiment score per day
    'Post Content': 'count'             # Counts total reviews per day
}).rename(columns={
    'Numeric_sentiment': 'avg_sentiment', 
    'Post Content': 'review_count'
}).fillna(0)

##### Saving to file

In [54]:
ts_out_path = f"{PROCESSED_DIR}timeseries_daily_social_media.csv"
daily_sentiment.to_csv(ts_out_path)

print(f"Time Series data saved to {ts_out_path}")

Time Series data saved to ../data/processed/timeseries_daily_social_media.csv
